# PL Standings: Premier League ETL Pipeline

## Purpose
This notebook pulls the current **Premier League table** from a football data API, reshapes it into a clean tabular dataset, and loads it into a **PostgreSQL** table (`public.football_table`).

## Pipeline overview

| Stage | What happens | Output |
| --- | --- | --- |
| **1. Setup** | Install dependencies, load credentials from a `.env` file | Environment variables in memory |
| **2. Extract** | Call the RapidAPI "Free API Live Football Data" standings endpoint | Raw JSON payload |
| **3. Transform** | Pick the fields we need, rename them, build a DataFrame | `df` (one row per club) |
| **4. Load** | Check the DataFrame matches the target table, then insert the rows | Rows in `public.football_table` |

## Prerequisites
- Python 3 with: `requests`, `pandas`, `psycopg2-binary`, `python-dotenv`
- A RapidAPI key subscribed to *Free API Live Football Data*
- A reachable PostgreSQL database that already contains the table `public.football_table` (this notebook does **not** create it, see the schema below)
- A `.env` file holding the variables listed below

## Environment variables

| Variable | Used for |
| --- | --- |
| `API_KEY` | RapidAPI key, sent as the `x-rapidapi-key` header |
| `API_HOST` | RapidAPI host, sent as the `x-rapidapi-host` header |
| `POSTGRES_HOST` | Database server hostname |
| `POSTGRES_PORT` | Database port (normally `5432`) |
| `POSTGRES_USER` | Database user |
| `POSTGRES_PASSWORD` | Database password |
| `POSTGRES_DATABASE` | Database name |

> **Security:** never commit the `.env` file. Keep it in `.gitignore` and share only a `.env.example` with blank values.

## Target table schema
The insert statement expects these columns, in this order. Column names contain spaces and capitals, so they must always be **double-quoted** in SQL. The notebook does not contain the table's `CREATE TABLE` statement, so the types below are *inferred from the DataFrame* and should be checked against your actual table.

| Column | Meaning | Inferred type |
| --- | --- | --- |
| `ID` | Club ID from the API | integer |
| `Position` | League position (1 = top) | integer |
| `Team` | Club name | text |
| `Matches Played` | Games played | integer |
| `Matches Won` | Wins | integer |
| `Matches Drawn` | Draws | integer |
| `Matches Lost` | Losses | integer |
| `GF-GA` | Goals for and against as text, e.g. `15-5` | text |
| `Goal Difference` | Goals for minus goals against | integer |
| `Points` | League points | integer |

## How to run
1. Fill in your `.env` file.
2. Run the cells **top to bottom**. Each section depends on variables created earlier.
3. Check the status code (Extract) and the column-match check (Load) before the insert runs.

## 1. Setup

### 1.1 Install dependencies
Run this once per environment. `%pip` installs into the environment the notebook kernel is using, which is why it is preferred over calling `pip` from a terminal.

In [ ]:
# Install everything the notebook imports.
# - psycopg2-binary : PostgreSQL driver (pre-compiled, no build tools needed)
# - pandas          : DataFrame for the transform step
# - requests        : HTTP client for the API call
# - python-dotenv   : reads credentials from a .env file
%pip install psycopg2-binary pandas requests python-dotenv

# NOTE: the original notebook also had a cell containing `py -m pip install pandas`.
# That is a Windows terminal command, not Python, so it raises a SyntaxError inside
# a notebook cell. It was redundant with the line above and has been removed.

### 1.2 Import libraries

In [ ]:
import os          # read environment variables
import json        # pretty-print and re-parse JSON
import requests    # call the football API
import psycopg2    # talk to PostgreSQL
from dotenv import load_dotenv  # load variables from a .env file
import pandas      # build and inspect the standings table

### 1.3 Load credentials from `.env`
`load_dotenv()` looks for a `.env` file by walking up from the current directory and copies its entries into `os.environ`. It returns `True` if it found a file.

If it returns `False` or the variables below come back as `None`, the notebook was started from a folder that cannot see your `.env`. The next cells help diagnose that.

In [ ]:
# Try the default lookup first. Returns True if a .env file was found and loaded.
load_dotenv()

In [ ]:
# Diagnostic: show exactly which .env file python-dotenv would pick up.
# An empty string means none was found.
from dotenv import find_dotenv

print(find_dotenv())

In [ ]:
# Fallback: load the .env file from an explicit path.
# NOTE: this path is specific to the original author's machine (WSL).
# Change it to wherever your own .env lives before running.
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path("/home/user/PL-Standings/.env")

print("File exists:", env_path.exists())   # sanity check before loading
print("File path:", env_path)

# override=True lets values from this file replace any variables already set
# in the environment, which avoids stale values from an earlier session.
load_dotenv(dotenv_path=env_path, override=True)

API_HOST = os.getenv("API_HOST")

# repr() makes a missing value show as None and exposes stray spaces or quotes.
print("Loaded API_HOST:", repr(API_HOST))

## 2. Extract
Fetch the raw standings from the API.

### 2.1 Configure the request
- **Endpoint:** `football-get-standing-all` returns the full league table.
- **`leagueid`:** `47` is the Premier League's ID in this API.
- **Headers:** RapidAPI requires the key and host on every request.

In [ ]:
# Credentials come from the .env file loaded above.
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")

# Premier League ID used by this API. Change it to pull a different league.
LEAGUE_ID = 47

# Confirm the host loaded correctly (never print the key itself).
print(API_HOST)

In [ ]:
# Endpoint that returns standings for every club in a league.
url = "https://free-api-live-football-data.p.rapidapi.com/football-get-standing-all"

# RapidAPI authentication headers.
headers = {
    "x-rapidapi-key": API_KEY,
    "x-rapidapi-host": API_HOST
}

# Query-string parameters, sent as ?leagueid=47
querystring = {
    "leagueid": LEAGUE_ID
}

### 2.2 Send the request
`requests.get` sends an HTTP GET. The response is kept in `response` so we can inspect the status code and body in the following cells.

In [ ]:
response = requests.get(url=url, headers=headers, params=querystring)

### 2.3 Check the response
A status code of **200** means success. Anything else (401 bad key, 403 not subscribed, 429 rate-limited, 5xx server error) means the data below cannot be trusted.

In [ ]:
# Status, raw body and the final URL (with query string) that was requested.
print("Status code:", response.status_code)
print("Response:", response.text)
print("Request URL:", response.url)

In [ ]:
# Parse the JSON body into Python dicts and lists.
payload = response.json()

# Displaying the variable lets us eyeball the structure.
payload

In [ ]:
# Pretty-print the whole payload (indent=5) to see how it is nested.
# Structure: {"status": ..., "response": {"standing": [ {club}, {club}, ... ]}}
formatted_response = json.dumps(payload, indent=5)
print(formatted_response)

### 2.4 Isolate the standings list
The club records sit under `payload["response"]["standing"]`, a list with **one dictionary per club**. Each club dictionary has these fields (the ones this notebook uses are marked):

| API field | Used as | Meaning |
| --- | --- | --- |
| `id` | `ID` | club ID |
| `idx` | `Position` | table position |
| `name` | `Team` | club name |
| `played`, `wins`, `draws`, `losses` | `Matches ...` | match record |
| `scoresStr` | `GF-GA` | goals for-against, e.g. `"15-5"` |
| `goalConDiff` | `Goal Difference` | goal difference |
| `pts` | `Points` | league points |

Fields returned but **not** used: `shortName`, `pageUrl`, `deduction`, `ongoing`, `qualColor`.

In [ ]:
# Drill down to the list of club records.
standings = payload['response']['standing']

In [ ]:
# Serialise the list back to a JSON string with readable indentation.
formatted_standing_list = json.dumps(standings, indent=4)

In [ ]:
# Inspect the standings list before transforming it.
print(formatted_standing_list)

## 3. Transform
Turn the nested JSON into a flat table: one row per club, with clear column names that match the database table.

### 3.1 Define the target columns
The order here is the order of the values in each row, and it must match the `INSERT` statement in the Load section.

In [ ]:
columns = [
    "ID",
    "Position",
    "Team",
    "Matches Played",
    "Matches Won",
    "Matches Drawn",
    "Matches Lost",
    "GF-GA",              # goals for - goals against, kept as text (e.g. "15-5")
    "Goal Difference",
    "Points"
]

### 3.2 Extract the fields from each club
Loop through every club dictionary, read the fields we need, and store them as a tuple in `rows`.

`formatted_standing_list` is currently a JSON *string*, so `json.loads` converts it back into a Python list first. (This dump-then-reload step is not strictly needed, because `standings` is already a list, but it is kept as originally written.)

In [ ]:
rows = []

# Convert the JSON string back into a list of dictionaries.
formatted_standing_list = json.loads(formatted_standing_list)

for clubs in formatted_standing_list:
    # Read each field from the API's naming and give it a descriptive local name.
    id = clubs["id"]                    # note: shadows Python's built-in id()
    position = clubs["idx"]             # table position
    team = clubs['name']
    matches_played = clubs['played']
    matches_won = clubs['wins']
    matches_drawn = clubs['draws']
    matches_lost = clubs['losses']
    gf_ga = clubs['scoresStr']          # e.g. "15-5"
    goal_diff = clubs['goalConDiff']
    points = clubs['pts']

    # One tuple per club, in the SAME order as the `columns` list above.
    tuple_of_records = (id, position, team, matches_played, matches_won,
                        matches_drawn, matches_lost, gf_ga, goal_diff, points)
    rows.append(tuple_of_records)

In [ ]:
# Debug checks: confirm the type and content of the parsed list.
print(type(formatted_standing_list))

print(formatted_standing_list)

In [ ]:
# Debug check: the finished tuples that will become DataFrame rows.
print(rows)

### 3.3 Build and inspect the DataFrame
Expect **20 rows** (one per Premier League club) and **10 columns**. Counts and points should be integers; `Team` and `GF-GA` are text.

In [ ]:
df = pandas.DataFrame(rows, columns=columns)

In [ ]:
# Row count, non-null counts and data types for each column.
df.info()

In [ ]:
# First five rows as a quick visual check.
df.head()

## 4. Load
Write the DataFrame into PostgreSQL.

### 4.1 Read database settings
These come from the same `.env` file. Any value that prints as `None` was not found.

In [ ]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DATABASE = os.getenv("POSTGRES_DATABASE")

### 4.2 Open a connection
`psycopg2.connect` opens a session with the database. The connection is kept in `connector` and reused for every later cell, then closed in the last cell.

In [ ]:
connector = psycopg2.connect(
    host=POSTGRES_HOST,
    database=POSTGRES_DATABASE,
    user=POSTGRES_USER,
    password=POSTGRES_PASSWORD,
    port=POSTGRES_PORT
)

### 4.3 Confirm the target table exists
Queries PostgreSQL's `information_schema` to check that `public.football_table` is there. If this prints `False`, create the table before continuing, because the insert will fail.

In [ ]:
# Clear any aborted transaction left over from an earlier failed cell.
# (In psycopg2, one failed statement blocks all later ones until a rollback.)
connector.rollback()

# A cursor runs SQL statements and fetches results.
cursor = connector.cursor()

# EXISTS returns a single True/False value.
cursor.execute("""
    SELECT EXISTS (
       SELECT 1
       FROM information_schema.tables
       WHERE table_schema ='public'
       AND table_name = 'football_table'
       );
    """)

table_exists = cursor.fetchone()[0]

print("Table exists:", table_exists)

### 4.4 Validate the DataFrame against the table
Before inserting, compare the DataFrame's columns with the table's columns. A mismatch in names or order would cause an insert error or, worse, values landing in the wrong columns.

In [ ]:
# Read the table's column names in their defined order.
cursor.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema = 'public'
    AND table_name = 'football_table'
    ORDER BY ordinal_position;
""")

table_columns = []

for row in cursor.fetchall():
    table_columns.append(row[0])

print("PostgreSQL columns:", table_columns)

In [ ]:
# The DataFrame's column names, as a plain list.
df_columns = []

for column in df.columns:
    df_columns.append(column)

In [ ]:
print("Dataframe Columns: ", df_columns)

In [ ]:
# True only if names AND order are identical.
print("Columns Match ", df_columns==table_columns)

In [ ]:
# If they do not match, these set differences show which columns are the problem.
# Both should print set() (empty) when everything lines up.
missing_in_table = set(df_columns) - set(table_columns)

missing_in_dataframe = set(table_columns) - set(df_columns)

print("Missing in PostgreSQL table:", missing_in_table)
print("Missing in DataFrame:", missing_in_dataframe)

### 4.5 Insert the records
1. **`insert_query`** is a parameterised statement. Each `%s` is a placeholder that psycopg2 fills safely, which protects against SQL injection and handles quoting and types.
2. **`records`** converts the DataFrame into a list of plain tuples.
3. **`executemany`** runs the insert once per tuple, and **`commit`** makes the changes permanent. If anything fails, **`rollback`** undoes the whole batch so the table is never left half-loaded.

> **Re-running warning:** this notebook only inserts. Running the Load section a second time adds another 20 rows unless the table has a unique constraint (for example on `"ID"`) that rejects duplicates.

In [ ]:
# Column names are double-quoted because they contain spaces and capitals.
# The column order here must match the `columns` list in the Transform section.
insert_query = """
    INSERT INTO public.football_table (
        "ID",
        "Position",
        "Team",
        "Matches Played",
        "Matches Won",
        "Matches Drawn",
        "Matches Lost",
        "GF-GA",
        "Goal Difference",
        "Points"
    )
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
"""

In [ ]:
# Convert each DataFrame row into a plain tuple (no index, no column names).
records = []

for row in df.itertuples(index=False, name=None):
    records.append(row)

In [ ]:
try:
    # Insert every record using the parameterised query.
    cursor.executemany(insert_query, records)

    # Make the insert permanent.
    connector.commit()

    print("Data loaded successfully!")
    print("Records inserted:", cursor.rowcount)

except Exception as error:
    # Undo the partial insert so the table stays consistent, then report why.
    connector.rollback()
    print("Error loading data:", error)

### 4.6 Clean up
Always close the cursor and connection when finished. Open connections consume database resources and can block other users if left hanging.

In [ ]:
# Close cursor
cursor.close()
print("Cursor closed:", cursor.closed)

# Close database connection
connector.close()
print("Connection closed:", connector.closed)

## Known limitations and suggested improvements

| Area | Limitation | Suggested improvement |
| --- | --- | --- |
| **API errors** | The status code is printed but not enforced, so a failed call would crash later at `payload['response']['standing']` | Call `response.raise_for_status()` right after the request |
| **Timeouts** | `requests.get` has no timeout and could hang indefinitely | Pass `timeout=30` (and consider retries for 429/5xx) |
| **Repeat runs** | Each run appends 20 more rows | Add a unique key on `"ID"` and use `INSERT ... ON CONFLICT DO UPDATE`, or truncate before loading |
| **No snapshot date** | The table has no column recording when the standings were captured, so history cannot be told apart | Add a `fetched_at` timestamp column |
| **Hard-coded path** | The fallback `.env` path only works on the original machine | Remove it once `load_dotenv()` finds the file, or read the path from an argument |
| **Cleanup on failure** | If a cell fails before the last one, the connection stays open | Wrap the Load stage in `try/finally` or use `with` blocks |
| **Notebook form** | Debug prints and manual cell order make it hard to schedule | Move the logic into functions or a `.py` script once it is stable |
| **Credentials** | Secrets live in a local `.env` file | Fine for development; use a secrets manager for shared or production use |